# 01 - Q Learning with Gymnasium

## Block 2: Understanding Q-Learning Intuitively

For this tutorial, we'll use Q-learning to solve the *Blackjack* environment. But first, let's understand how Q-learning works conceptually.

Q-learning builds a giant “cheat sheet” called a **Q-table** that tells the agent how good each action is in each situation:



* **Rows:** Different situations (states) the agent can encounter.
* **Columns:** Different actions the agent can take.
* **Values:** How good that action is in that situation (the expected future reward).

**In the context of Blackjack:**
* **States:** Your hand value, the dealer's showing card, and whether you have a usable ace.
* **Actions:** Hit (take another card) or Stand (keep your current hand).
* **Q-values:** The expected reward for choosing a specific action in a specific state.

---

### 1. The Learning Process

How does the agent actually fill out this Q-table? It learns through trial and error:

1.  **Interact:** Try an action and observe what happens (receive a reward and the new state).
2.  **Update:** Update the "cheat sheet" (Q-table) based on the outcome: *"That action was better/worse than I thought."*
3.  **Iterate:** Gradually improve by trying actions and continuously updating estimates.
4.  **Balance:** Manage the **Exploration vs. Exploitation** tradeoff (trying new things vs. using what is already known to work).

> **Why it works:** Over time, good actions accumulate higher Q-values, and bad actions get lower Q-values. Eventually, the agent learns to simply pick the actions with the highest expected rewards.

---

### 2. About the Environment: Blackjack



Blackjack is one of the most popular casino card games and is perfect for learning RL because it has clear rules, simple observations, discrete actions, and immediate feedback.

*Note: This environment uses an infinite deck (cards are drawn with replacement), so card counting won't work. The agent must learn the optimal basic strategy purely through trial and error.*

#### Environment Details

| Feature | Description |
| :--- | :--- |
| **Observation Space** | A tuple: `(player_sum, dealer_card, usable_ace)` |
| *player_sum* | Current hand value (4 to 21) |
| *dealer_card* | Dealer's face-up card (1 to 10) |
| *usable_ace* | Whether the player has a usable ace (`True` or `False`) |
| **Action Space** | `0` = Stand, `1` = Hit |
| **Rewards** | `+1` for a win, `-1` for a loss, `0` for a draw |
| **Termination** | Episode ends when the player stands or busts (goes over 21) |

📚 *Reference:* [Gymnasium Blackjack Environment Documentation](https://gymnasium.farama.org/environments/toy_text/blackjack/)

---

### 3. Building a Q-Learning Agent

Let's build our agent step by step in the code below. We will need functions for:

* **Choosing actions:** Implementing the exploration vs. exploitation strategy.
* **Learning from experience:** Updating our Q-values based on results.
* **Managing exploration:** Gradually reducing randomness over time.

### 4. Coding the Q-Learning Agent (Pseudocode)

Now it is your turn to build the agent! We have created a skeleton class for you. Your job is to fill in the logic for each method based on the Q-learning algorithm.

Here is a blueprint to guide you:

#### The Math: Bellman Equation
Before writing the code, recall the core Q-learning update formula. This is how the agent updates its "cheat sheet":
$$Q(s,a) = Q(s,a) + \alpha \cdot [r + \gamma \max_{a'} Q(s', a') - Q(s,a)]$$
*Where $\alpha$ is the learning rate, $\gamma$ is the discount factor, $r$ is the reward, and $s'$ is the next state.*



---

#### 1. `__init__` (Initialization)
**Goal:** Set up the agent's memory and hyperparameters.
* Store the environment (`env`), learning rate, discount factor, and epsilon parameters as class variables.
* **The Q-Table:** Create a dictionary to act as the Q-table.
    * *Hint:* Use Python's `collections.defaultdict`. If the agent sees a new state, it should automatically create an entry with an array of zeros (one zero for each possible action).
* Create an empty list called `training_error` to track our temporal difference errors (useful for debugging later).

#### 2. `get_action(state)` (Epsilon-Greedy Strategy)
**Goal:** Decide whether to explore or exploit.

* Generate a random decimal number between 0 and 1.
* **IF** the random number is less than `self.epsilon`:
    * *Explore:* Return a random action sampled from the environment's action space.
* **ELSE**:
    * *Exploit:* Look up the current `state` in the Q-table. Find the action (index) that currently has the highest expected reward. Return that action.
    * *Hint:* Use `np.argmax()` to find the index of the highest value in a Numpy array.

#### 3. `update(state, action, reward, terminated, next_state)`
**Goal:** Apply the Bellman equation to update the Q-value for the action just taken.
* **Step A (Future Value):** Look at the `next_state` in the Q-table and find the maximum possible Q-value.
    * *Important:* If `terminated` is True, the episode is over, so the future value must be `0`!
* **Step B (Target):** Calculate the target value: `reward + (discount_factor * future_value)`.
* **Step C (Temporal Difference):** Calculate the error between the target and our *current* estimate: `target - current_q_value`.
* **Step D (Update):** Update the Q-table for the current `state` and `action`:
    `new_q = current_q + (learning_rate * temporal_difference)`.
* Append the `temporal_difference` to your `training_error` list.

#### 4. `decay_epsilon()`
**Goal:** Slowly reduce the exploration rate so the agent relies more on its learned knowledge over time.
* Subtract `self.epsilon_decay` from `self.epsilon`.
* Ensure that `self.epsilon` never drops below `self.final_epsilon` (you can use Python's `max()` function for this).

In [ ]:
from collections import defaultdict
import gymnasium as gym
import numpy as np


class BlackjackAgent:
    def __init__(
        self,
        env: gym.Env,
        learning_rate: float,
        initial_epsilon: float,
        epsilon_decay: float,
        final_epsilon: float,
        discount_factor: float = 0.95,
    ):
        """Initialize a Q-Learning agent.

        Args:
            env: The training environment
            learning_rate: How quickly to update Q-values (0-1)
            initial_epsilon: Starting exploration rate (usually 1.0)
            epsilon_decay: How much to reduce epsilon each episode
            final_epsilon: Minimum exploration rate (usually 0.1)
            discount_factor: How much to value future rewards (0-1)
        """
        self.env = env

        # Q-table: maps (state, action) to expected reward
        # defaultdict automatically creates entries with zeros for new states
        self.q_values = defaultdict(lambda: np.zeros(env.action_space.n))

        self.lr = learning_rate
        self.discount_factor = discount_factor  # How much we care about future rewards

        # Exploration parameters
        self.epsilon = initial_epsilon
        self.epsilon_decay = epsilon_decay
        self.final_epsilon = final_epsilon

        # Track learning progress
        self.training_error = []

    def get_action(self, obs: tuple[int, int, bool]) -> int:
        """Choose an action using epsilon-greedy strategy.

        Returns:
            action: 0 (stand) or 1 (hit)
        """
        # TODO: With probability epsilon: explore (random action)
        if ...:
          action = ...
        else:
          action = ...
        return action
        # TODO: With probability (1-epsilon): exploit (best known action)

    def update(
        self,
        obs: tuple[int, int, bool],
        action: int,
        reward: float,
        terminated: bool,
        next_obs: tuple[int, int, bool],
    ):
        """Update Q-value based on experience.

        This is the heart of Q-learning: learn from (state, action, reward, next_state)
        """
        # TODO: What's the best we could do from the next state?
        # (Zero if episode terminated - no future rewards possible)
        future_q_value = ...

        # # TODO: What should the Q-value be as target? (Bellman equation, right side)
        target = ...

        # TODO: How wrong was our current estimate of next step?
        temporal_difference = ...

        # TODO: Update our estimate in the direction of the error
        # Learning rate controls how big steps we take
        self.q_values[obs][action] = ...

        # Track learning progress (useful for debugging)
        self.training_error.append(temporal_difference)

    def decay_epsilon(self):
        """Reduce exploration rate after each episode."""
        self.epsilon = max(self.final_epsilon, self.epsilon - self.epsilon_decay)

In [ ]:
# Training hyperparameters
learning_rate = 0.01        # How fast to learn (higher = faster but less stable)
n_episodes = 50_000        # Number of hands to practice
start_epsilon = 1.0         # Start with 100% random actions
epsilon_decay = start_epsilon / (n_episodes / 2)  # Reduce exploration over time
final_epsilon = 0.1         # Always keep some exploration

# Create environment and agent
env = gym.make("Blackjack-v1", sab=False, render_mode="rgb_array")
print(env.action_space)
print(env.observation_space)
env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes)

agent = BlackjackAgent(
    env=env,
    learning_rate=learning_rate,
    initial_epsilon=start_epsilon,
    epsilon_decay=epsilon_decay,
    final_epsilon=final_epsilon,
)

In [ ]:
from tqdm import tqdm  # Progress bar

for episode in tqdm(range(n_episodes)):
    # Start a new hand
    obs, info = env.reset()
    done = False

    # Play one complete hand
    while not done:
        # TODO: Agent chooses action
        action = ...
        # TODO: Take action and observe result
        ... = env.step(...)
        # TODO: Learn from this experience (update the agent)
        agent.update(...)
        # Move to next state
        done = terminated or truncated
        obs = next_obs # important/hint: you must have a "next_obs" variable before this line

    # Reduce exploration rate (agent becomes less random over time)
    agent.decay_epsilon()

In [ ]:
# Run the code below to plot the training metrics in time

In [ ]:
# @title
from matplotlib import pyplot as plt

def get_moving_avgs(arr, window, convolution_mode):
    """Compute moving average to smooth noisy data."""
    return np.convolve(
        np.array(arr).flatten(),
        np.ones(window),
        mode=convolution_mode
    ) / window

# Smooth over a 500-episode window
rolling_length = 500
fig, axs = plt.subplots(ncols=3, figsize=(12, 5))

# Episode rewards (win/loss performance)
axs[0].set_title("Episode rewards")
reward_moving_average = get_moving_avgs(
    env.return_queue,
    rolling_length,
    "valid"
)
axs[0].plot(range(len(reward_moving_average)), reward_moving_average)
axs[0].set_ylabel("Average Reward")
axs[0].set_xlabel("Episode")

# Episode lengths (how many actions per hand)
axs[1].set_title("Episode lengths")
length_moving_average = get_moving_avgs(
    env.length_queue,
    rolling_length,
    "valid"
)
axs[1].plot(range(len(length_moving_average)), length_moving_average)
axs[1].set_ylabel("Average Episode Length")
axs[1].set_xlabel("Episode")

# Training error (how much we're still learning)
axs[2].set_title("Training Error")
training_error_moving_average = get_moving_avgs(
    agent.training_error,
    rolling_length,
    "same"
)
axs[2].plot(range(len(training_error_moving_average)), training_error_moving_average)
axs[2].set_ylabel("Temporal Difference Error")
axs[2].set_xlabel("Step")

plt.tight_layout()
plt.show()

### Visualize your trained agent!

In [ ]:
!pip install pyvirtualdisplay

In [ ]:
# Run the code below to test your agent win rate!

In [ ]:
# @title
# Test the trained agent
import glob
import io
import base64
from IPython.display import HTML
from IPython import display as ipythondisplay
from pyvirtualdisplay import Display
from gymnasium.wrappers import RecordVideo


def test_agent(agent, env, num_episodes=100):
    """Test agent performance without learning or exploration."""
    total_rewards = []
    env = RecordVideo(env, video_folder="./expert_video", name_prefix="sb3_agent", episode_trigger=lambda x: True)

    # Temporarily disable exploration for testing
    old_epsilon = agent.epsilon
    agent.epsilon = 0.0  # Pure exploitation

    for _ in range(num_episodes):
        obs, info = env.reset()
        episode_reward = 0
        done = False

        while not done:
            action = agent.get_action(obs)
            obs, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward
            done = terminated or truncated

        total_rewards.append(episode_reward)

    # Restore original epsilon
    agent.epsilon = old_epsilon

    win_rate = np.mean(np.array(total_rewards) > 0)
    average_reward = np.mean(total_rewards)

    print(f"Test Results over {num_episodes} episodes:")
    print(f"Win Rate: {win_rate:.1%}")
    print(f"Average Reward: {average_reward:.3f}")
    print(f"Standard Deviation: {np.std(total_rewards):.3f}")


# Test your agent
# 1. Start a virtual display (Required for Colab)
display = Display(visible=0, size=(1400, 900))
display.start()
env = gym.make("Blackjack-v1", sab=False, render_mode="rgb_array")
test_agent(agent, env)


In [ ]:
# Run the code below to show a video of your agent!

In [ ]:
# @title
# 5. Helper function to display the video in Colab
def show_video(folder_path):
    mp4list = glob.glob(f'{folder_path}/*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0] # Grab the first video in the folder
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        ipythondisplay.display(HTML(data='''<video alt="test" autoplay
                    loop controls style="height: 400px;">
                    <source src="data:video/mp4;base64,{0}" type="video/mp4" />
                 </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video. Did the environment close properly?")

# Display our recording!
show_video('./expert_video')

### Print the agent's q-values (first 10 values only): Can you understand what these numbers represent?

In [ ]:
# TODO: Print the agent's q-values (first 10 values only): Can you understand what these numbers represent?

In [ ]:
# Run the code below to show a heatmap of the q values!

In [ ]:
# @title
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_blackjack_policy(agent_q_values):
    """
    Plots the policy (Hit or Stick) learned by the agent.
    0 = Stick (Red)
    1 = Hit (Green)
    """
    # Create empty grids for our heatmaps:
    # Y-axis: Player Sum (11 to 21) -> 11 rows
    # X-axis: Dealer Card (1 to 10) -> 10 columns
    # We initialize with -1 so we can see if any states were never visited
    policy_usable_ace = np.full((11, 10), -1)
    policy_no_usable_ace = np.full((11, 10), -1)

    # Fill the grids with the best action from the Q-table
    for player_sum in range(11, 22):
        for dealer_card in range(1, 11):
            # Row and Column indices for our numpy array
            row = player_sum - 11
            col = dealer_card - 1

            # --- For Usable Ace (1) ---
            state_usable = (player_sum, dealer_card, 1)
            if state_usable in agent_q_values:
                # np.argmax picks 0 (Stick) or 1 (Hit)
                policy_usable_ace[row, col] = np.argmax(agent_q_values[state_usable])

            # --- For No Usable Ace (0) ---
            state_no_usable = (player_sum, dealer_card, 0)
            if state_no_usable in agent_q_values:
                policy_no_usable_ace[row, col] = np.argmax(agent_q_values[state_no_usable])

    # --- Plotting the Heatmaps ---
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # Custom color map: Red for Stick (0), Green for Hit (1), Gray for Unvisited (-1)
    cmap = sns.color_palette(["gray", "lightcoral", "lightgreen"])

    # Plot 1: Usable Ace
    sns.heatmap(policy_usable_ace,
                vmin=-1, vmax=1, # <--- ADD THIS LINE HERE
                ax=axes[0], cmap=cmap, cbar=False,
                xticklabels=range(1, 11), yticklabels=range(11, 22),
                linewidths=1, linecolor='white')
    axes[0].set_title("Strategy With Usable Ace", fontsize=14)
    axes[0].set_xlabel("Dealer Showing")
    axes[0].set_ylabel("Player Sum")
    axes[0].invert_yaxis() # Put 21 at the top

    # Plot 2: No Usable Ace
    sns.heatmap(policy_no_usable_ace,
                vmin=-1, vmax=1, # <--- ADD THIS LINE HERE
                ax=axes[1], cmap=cmap, cbar=False,
                xticklabels=range(1, 11), yticklabels=range(11, 22),
                linewidths=1, linecolor='white')
    axes[1].set_title("Strategy Without Usable Ace", fontsize=14)
    axes[1].set_xlabel("Dealer Showing")
    axes[1].set_ylabel("Player Sum")
    axes[1].invert_yaxis() # Put 21 at the top

    # Add a custom legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='lightgreen', label='Hit (1)'),
                       Patch(facecolor='lightcoral', label='Stick (0)')]
    fig.legend(handles=legend_elements, loc='lower center', ncol=2, fontsize=12, bbox_to_anchor=(0.5, -0.05))

    plt.tight_layout()
    plt.show()

# Run the function using your agent's dictionary!
# Assuming your dictionary is named agent.q_values
plot_blackjack_policy(agent.q_values)